# Saheli — Evaluation Benchmark (Colab)

Runs the three-row benchmark on 20 maternal health test cases:
- Row 1: Rule-only baseline
- Row 2: Pure LLM (fine-tune in isolation)
- Row 3: LLM + Rule safety-net (production path)

**Runtime:** T4 GPU. Takes ~5 minutes.

**Before running:** Upload `saheli.zip` (project folder zipped, without the models/ folder).

## Step 1 — Install dependencies

In [ ]:
# Install llama-cpp-python with CUDA support (prebuilt wheel for Colab T4 — CUDA 12.2)
!pip install -q --upgrade pip
!pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
    --upgrade --quiet

!pip install -q scikit-learn geopy datasets

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

# Verify llama-cpp-python has CUDA
from llama_cpp import llama_cpp
import ctypes
print('llama-cpp CUDA:', hasattr(llama_cpp, 'llama_backend_init'))

## Step 2 — Upload project zip

In [ ]:
from google.colab import files
import os, sys, zipfile, shutil

print('Upload saheli.zip (without the models/ folder — GGUF will be downloaded separately)')
uploaded = files.upload()

WORKDIR = '/content/saheli'
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR)

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(WORKDIR)

# Flatten if extra top-level folder
entries = os.listdir(WORKDIR)
if len(entries) == 1 and os.path.isdir(f'{WORKDIR}/{entries[0]}'):
    inner = f'{WORKDIR}/{entries[0]}'
    for e in os.listdir(inner):
        shutil.move(f'{inner}/{e}', f'{WORKDIR}/{e}')
    os.rmdir(inner)

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)

print('Working dir:', os.getcwd())
print('evaluate_model.py:', os.path.exists('finetune/evaluate_model.py'))
print('sample_cases.json :', os.path.exists('tests/sample_cases.json'))

## Step 3 — Download GGUF from HuggingFace

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import os

HF_TOKEN   = userdata.get('HF_TOKEN')
HF_REPO    = 'sriramarivazhagan/saheli-gemma4-e4b-v3'
GGUF_FILE  = 'gemma-4-e4b-it.Q4_K_M.gguf'
MODEL_DIR  = '/content/saheli/models'

os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Downloading {GGUF_FILE} from {HF_REPO}...')
path = hf_hub_download(
    repo_id   = HF_REPO,
    filename  = GGUF_FILE,
    token     = HF_TOKEN,
    local_dir = MODEL_DIR,
)
print(f'Downloaded to: {path}  ({os.path.getsize(path)/1e9:.2f} GB)')

## Step 4 — Patch settings and run evaluation

In [ ]:
# Patch settings.py to point at the downloaded GGUF and use smaller context
settings_path = f'{WORKDIR}/config/settings.py'
with open(settings_path) as f:
    src = f.read()

src = src.replace(
    src[src.find('MODEL_PATH'):src.find('\n', src.find('MODEL_PATH'))],
    f"MODEL_PATH = '{MODEL_DIR}/{GGUF_FILE}'"
).replace('CONTEXT_LENGTH = 8192', 'CONTEXT_LENGTH = 2048')

with open(settings_path, 'w') as f:
    f.write(src)

print('Patched settings.py:')
print(open(settings_path).read())

# Quick GPU sanity check — confirm n_gpu_layers=-1 actually uses CUDA
print('\n--- GPU offload check ---')
import torch
print(f'VRAM before model load: {torch.cuda.memory_allocated()/1e9:.2f} GB used')
from llama_cpp import Llama
_test = Llama(model_path=f'{MODEL_DIR}/{GGUF_FILE}', n_ctx=512, n_gpu_layers=-1, verbose=True)
print(f'VRAM after  model load: {torch.cuda.memory_allocated()/1e9:.2f} GB used')
del _test
print('GPU offload confirmed ✓ — proceed to Step 5')

In [ ]:
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = WORKDIR
env['PYTHONUNBUFFERED'] = '1'   # force line-buffered stdout

proc = subprocess.Popen(
    [sys.executable, '-u', 'finetune/evaluate_model.py'],
    cwd=WORKDIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

stderr_lines = []
import threading

def _drain_stderr():
    for line in proc.stderr:
        line = line.rstrip()
        if line:
            stderr_lines.append(line)

t = threading.Thread(target=_drain_stderr, daemon=True)
t.start()

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
t.join()

if stderr_lines:
    print('\nSTDERR (last 40 lines):')
    print('\n'.join(stderr_lines[-40:]))